# Chapter 28: Curvature as a Force between Neighbouring Geodesics


## Source Span And Scope

    This notebook covers Chapter 28, **Curvature as a Force between Neighbouring Geodesics**, with source orientation at printed pages 269-279 and PDF pages 298-308. It is intentionally written as standalone course material. The chapter's role is: Interprets curvature dynamically through geodesic deviation and the Jacobi equation. The source book is visual and dramatic in style; this edition keeps that spirit by replacing passive reading with computations, diagrams, and checks that can be rerun.

    The notebook does not reproduce the book's prose or figures. Instead it reconstructs the mathematical path in fresh language. When a diagram in the book carries an argument, the notebook turns the same idea into a small model: a curve whose curvature can be measured, a surface whose metric can be differentiated, a loop whose holonomy can be estimated, or a differential form whose exterior derivative can be checked symbolically.

    ## Translation Guide

    The translation from page to notebook is guided by these chapter themes:

    - Jacobi equation for nearby geodesics
- Zero curvature and parallel behavior in the plane
- Positive curvature and convergence on the sphere
- Negative curvature and divergence on the pseudosphere
- Geodesic polar coordinate proof
- Relative acceleration as velocity holonomy
- Small geodesic-circle circumference and area

    Computationally, these themes become four habits. First, every geometric claim gets an object that can be inspected: a parametrized curve, a chart, a metric, a frame, a form, or a tensor. Second, every drawing gets a numerical shadow: length, angle, area, curvature, index, period, or determinant. Third, every formula is tested in a friendly special case before it is trusted in a general setting. Fourth, every apparent coordinate trick is pushed back toward its invariant meaning.

    ## Route

    The route is deliberately layered. We begin with the geometric question that motivates the chapter. We then identify the minimum computational representation needed to hold that question. After that, we work one or two examples where the formulas are small enough to audit by eye. The final cells save artifacts under the book-local `artifacts` directory and assert that the most important identities survived execution.

    A reader should finish with three kinds of understanding. Conceptual understanding: what the chapter says about geometry. Operational understanding: how to compute the objects involved. Diagnostic understanding: what would go wrong if a sign, scale factor, orientation, or coordinate interpretation were mistaken.

    ## Concept Map

    The chapter can be read as an answer to a single organizing question: how can we see interprets curvature dynamically through geodesic deviation and the jacobi equation.? The answer is not merely a formula. The formula is the compact end point of a geometric story. For that reason the notebook repeatedly moves between pictures, algebra, and tests. A picture suggests what should be true; algebra compresses the statement; code checks whether the compression still behaves like the picture.

    The major concepts are Jacobi equation for nearby geodesics; Zero curvature and parallel behavior in the plane; Positive curvature and convergence on the sphere; Negative curvature and divergence on the pseudosphere; Geodesic polar coordinate proof; Relative acceleration as velocity holonomy; Small geodesic-circle circumference and area. These are not separate vocabulary items. They form a chain. Earlier ideas provide the measurement language for later ones, and later ideas explain why the earlier definitions were chosen so carefully. For example, curvature is not just a number attached to a curve or surface; it is the obstruction to certain flat expectations. Transport is not just a rule for dragging arrows; it is a way to ask whether the space itself has twisted the arrow. Forms are not just symbolic decorations such as `dx` and `dy`; they are machines for measuring directed pieces of geometry.

    ## Computational Lens

    The computations in this notebook favor transparent formulas over speed. A symbolic expression from SymPy is useful because it can still be read. A small NumPy array is useful because its entries can be compared with the drawing. A saved JSON check is useful because it records the invariant without burying it in notebook output. This is the same reason the utilities in `utils/` are intentionally small: they are teaching tools first.

    The central safety rule is to keep track of what kind of object is being handled. Coordinates are not points; they are labels assigned by a chart. A metric is not a drawing; it is the rule that converts coordinate changes into lengths and angles. A tangent vector is not a free arrow in space once it lives on a curved surface; it belongs to a particular tangent plane. A form is not a vector with a different costume; it acts on vectors and measures them. Many mistakes in differential geometry come from letting these distinctions blur.

    ## Pitfalls

    The common traps are predictable. A parameter may not be arc length. A visually circular path may not be a geodesic circle. A curve that bends in ambient space may still be intrinsically straight on a surface. A coordinate basis may stretch or skew, making Euclidean-looking formulas false. A sign in an oriented area or wedge product may encode real orientation rather than a cosmetic convention. A tensor component table may change under a basis change even though the tensor itself has not changed.

    The notebook counters these traps by asking for invariants. Does the spherical triangle area agree with angular excess? Does the metric of a unit sphere produce curvature one? Does `d(d(omega))` vanish for a form? Does a saved artifact exist exactly where the chapter says it should? These checks are small, but they keep the exposition honest.

    ## Applied Lab

    Lab focus: Build an executable check for interprets curvature dynamically through geodesic deviation and the jacobi equation. The lab is intentionally modest. Its goal is not to automate all of differential geometry; it is to create one reliable computational handle on the chapter. Once the handle works, the reader can change inputs, rerun cells, and watch which features remain invariant.

    ## Standalone Coverage Contract

    This notebook is designed to stand on its own even though it respects the book's order. That means the reader should not need to remember a picture from the PDF in order to follow the argument here. Any visual claim is restated as a construction, any construction is paired with the data it preserves, and any preserved data is checked in code. The coverage is broad rather than terse: historical motivation is included when it explains why a definition exists, computational detail is included when it prevents a false shortcut, and examples are chosen because they expose the geometry rather than because they are algebraically flashy.

    The notebook also makes its limits explicit. It does not try to replace a full research text on the topic, and it does not pretend that one symbolic example proves every theorem. Instead it gives a durable working model: the reader can identify the objects, compute with them in ordinary cases, recognize the invariants, and know which later chapters depend on the idea. That is the practical meaning of complete chapter coverage for this course.

    ## Takeaways

    The chapter's durable lesson is that geometry becomes clearer when formulas are treated as records of visual and operational facts. The notebook therefore leaves each section with a definition, a picture or artifact, and a check. If all three agree, the idea is no longer only a line in a book; it has become something the reader can use.


In [ ]:
from pathlib import Path
import sys
import json
import math

import matplotlib.pyplot as plt
import numpy as np
import sympy as sp

BOOK_ROOT = Path.cwd()
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the VDGF book root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import save_json, save_matplotlib, save_text, display_artifact
from utils.visuals import build_chapter_visual
from utils.dg import (
    gaussian_curvature_2d,
    hyperbolic_distance,
    metric_tensor,
    osculating_circle,
    plane_curve_curvature,
    poincare_disk_distance,
    poincare_disk_to_hyperboloid,
    shape_operator,
    sphere_embedding,
)
from utils.forms import CoordinateSystem, basis_form, d, evaluate, hodge_star, line_integral

ARTIFACT_TOPIC = "chapter-28"
ARTIFACT_BASE = BOOK_ROOT / "artifacts"
np.set_printoptions(precision=4, suppress=True)
print(f"Book root: {BOOK_ROOT.name}")


In [ ]:
source_span = json.loads(r'''
{
  "label": "Chapter 28",
  "title": "Curvature as a Force between Neighbouring Geodesics",
  "printed_pages": "269-279",
  "pdf_pages": "298-308",
  "focus": "Interprets curvature dynamically through geodesic deviation and the Jacobi equation.",
  "topics": [
    "Jacobi equation for nearby geodesics",
    "Zero curvature and parallel behavior in the plane",
    "Positive curvature and convergence on the sphere",
    "Negative curvature and divergence on the pseudosphere",
    "Geodesic polar coordinate proof",
    "Relative acceleration as velocity holonomy",
    "Small geodesic-circle circumference and area"
  ]
}
''')
path = save_json(source_span, ARTIFACT_TOPIC, "checks", "source-span.json", root=ARTIFACT_BASE)
print(path.relative_to(BOOK_ROOT))
assert path.exists()


In [ ]:
theta = np.linspace(0, 2 * np.pi, 240)
velocity_circle = np.column_stack([-np.sin(theta), np.cos(theta)])
acceleration_circle = np.column_stack([-np.cos(theta), -np.sin(theta)])
kappa_circle = plane_curve_curvature(velocity_circle, acceleration_circle)

parabola_t = np.linspace(-1.5, 1.5, 160)
velocity_parabola = np.column_stack([np.ones_like(parabola_t), parabola_t])
acceleration_parabola = np.column_stack([np.zeros_like(parabola_t), np.ones_like(parabola_t)])
kappa_parabola = plane_curve_curvature(velocity_parabola, acceleration_parabola)
circle = osculating_circle(np.array([1.0, 0.0]), np.array([0.0, 1.0]), np.array([-1.0, 0.0]))

u, v = sp.symbols("u v", positive=True)
sphere = sphere_embedding(u, v)
sphere_metric = metric_tensor(sphere, [u, v])
sphere_K = sp.simplify(gaussian_curvature_2d(sphere_metric, [u, v]))

curvature_checks = {
    "unit_circle_curvature_mean": float(np.mean(kappa_circle)),
    "unit_circle_curvature_std": float(np.std(kappa_circle)),
    "parabola_curvature_at_zero": float(kappa_parabola[len(kappa_parabola) // 2]),
    "unit_sphere_metric": str(sphere_metric),
    "unit_sphere_gaussian_curvature": str(sphere_K),
    "osculating_circle_at_zero": {
        "center": circle.center.tolist(),
        "radius": circle.radius,
        "curvature": circle.curvature,
    },
}
path = save_json(curvature_checks, ARTIFACT_TOPIC, "checks", "dg-symbolic-checks.json", root=ARTIFACT_BASE)
assert np.allclose(kappa_circle, 1.0)
assert sp.simplify(sphere_K - 1) == 0
print(path.relative_to(BOOK_ROOT))


In [ ]:
coords = CoordinateSystem("R3", "x y z")
x, y, z = coords.symbols
dx, dy, dz = [basis_form(coords, i) for i in range(3)]
omega = x * dy + y * dz + z * dx
two_form = d(omega)
closed_check = d(two_form)
area_form = dx.wedge(dy)
flux_dual = hodge_star(dx)
form_checks = {
    "omega": repr(omega),
    "d_omega": repr(two_form),
    "d_squared_zero": not bool(closed_check),
    "area_on_basis_vectors": str(evaluate(area_form, [1, 0, 0], [0, 1, 0])),
    "hodge_star_dx": repr(flux_dual),
}
path = save_json(form_checks, ARTIFACT_TOPIC, "checks", "forms-checks.json", root=ARTIFACT_BASE)
assert not bool(closed_check)
assert evaluate(area_form, [1, 0, 0], [0, 1, 0]) == 1
print(path.relative_to(BOOK_ROOT))


In [ ]:
visual_spec = json.loads('{\n  "filename": "jacobi-geodesic-deviation.png",\n  "family": "jacobi",\n  "title": "Jacobi Geodesic Deviation",\n  "caption": "Nearby geodesics focus or diverge according to the sign of curvature.",\n  "chapter": 28,\n  "entry_label": "Chapter 28",\n  "entry_title": "Curvature as a Force between Neighbouring Geodesics",\n  "source_focus": "Interprets curvature dynamically through geodesic deviation and the Jacobi equation."\n}')
visual_path, visual_metrics = build_chapter_visual(visual_spec, ARTIFACT_BASE, ARTIFACT_TOPIC)
display_artifact(visual_path, width=720)

assert visual_path.exists()
assert visual_metrics["file_size"] > 1000
assert visual_metrics["width"] >= 300 and visual_metrics["height"] >= 200
assert visual_metrics["pixel_std"] > 1.0

print(visual_path.relative_to(BOOK_ROOT))
print(json.dumps(visual_metrics, indent=2, sort_keys=True))


In [ ]:
# The octant triangle on the unit sphere has three right angles,
# so its spherical excess and area are pi/2.
triangle_area = math.pi / 2
p = poincare_disk_to_hyperboloid(np.array([0.0, 0.0]))
q = poincare_disk_to_hyperboloid(np.array([0.25, 0.0]))
hdist = hyperbolic_distance(p, q)
disk_distance = poincare_disk_distance(np.array([0.0, 0.0]), np.array([0.25, 0.0]))
summary = {
    "spherical_octant_area": triangle_area,
    "expected_octant_area": math.pi / 2,
    "hyperboloid_distance": hdist,
    "poincare_disk_distance": disk_distance,
    "visual_artifact": str(visual_path.relative_to(BOOK_ROOT)),
    "visual_title": visual_spec["title"],
    "visual_metrics": visual_metrics,
}
path = save_json(summary, ARTIFACT_TOPIC, "checks", "final-sanity.json", root=ARTIFACT_BASE)
assert abs(triangle_area - math.pi / 2) < 1e-9
assert abs(hdist - disk_distance) < 1e-9
assert visual_path.exists()
assert visual_metrics["pixel_std"] > 1.0
assert path.exists()
print(path.relative_to(BOOK_ROOT))
